# ODAC25 Phase 3 material benchmark

Runs one bounded ODAC25 training shard and the frozen 138-MOF validation pool. It evaluates training-only constant/composition baselines, ODAC25 eSEN, and UMA on identical paired CO2/H2O targets. Attach `MOF_DAC_Kaggle_bundle`, enable Internet and a T4 GPU, and expose a Kaggle secret named `HF_TOKEN` to this notebook.


In [ ]:
%pip install -q "huggingface-hub>=0.28" "fairchem-core==2.23.0" "fairchem-data-odac==0.1.0"


In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Enable the HF_TOKEN Kaggle secret"
archives = list(Path("/kaggle/input").rglob("MOF_DAC_Kaggle_bundle.zip"))
if len(archives) == 1:
    root = Path("/kaggle/working/phase3_code")
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as bundle:
        bundle.extractall(root)
else:
    hits = list(Path("/kaggle/input").rglob("kaggle/phase3_material_pipeline.py"))
    assert len(hits) == 1, "Attach exactly one current project bundle"
    root = hits[0].parents[1]
env = {**os.environ, "PYTHONPATH": str(root)}
assert (root / "kaggle/phase3_material_pipeline.py").is_file()
print("code root", root)


In [ ]:
from huggingface_hub import hf_hub_download

TRAIN_REPO = "colabfit/Open_Direct_Air_Capture_ODAC2025_Train_Filtered"
TRAIN_REVISION = "064de0077b3de4745f58c58e7a34da933c8b727e"
TRAIN_FILE = "co/co_9.parquet"
train_path = hf_hub_download(TRAIN_REPO, TRAIN_FILE, repo_type="dataset",
                             revision=TRAIN_REVISION, token=os.environ["HF_TOKEN"])
print(train_path, Path(train_path).stat().st_size)


In [ ]:
import requests

VAL_URL = "https://dl.fbaipublicfiles.com/dac/odac25/20250918/odac25_filtered_val.tar.gz"
wanted = {
    "val/mof_plus_adsorbate/part_00000.aselmdb",
    "val/mof/part_00000.aselmdb",
}
data_root = Path("/kaggle/working/phase3_val")
found = {}
with requests.get(VAL_URL, stream=True, timeout=(30, 600)) as response:
    response.raise_for_status()
    response.raw.decode_content = True
    with tarfile.open(fileobj=response.raw, mode="r|gz") as archive:
        for member in archive:
            if member.name not in wanted:
                continue
            destination = data_root / member.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            stream = archive.extractfile(member)
            assert stream is not None
            with stream, destination.open("wb") as sink:
                shutil.copyfileobj(stream, sink, length=8 * 1024 * 1024)
            found[member.name] = destination
            print("extracted", member.name, member.size)
            if set(found) == wanted:
                break
assert set(found) == wanted, found
target_path = found["val/mof_plus_adsorbate/part_00000.aselmdb"]
bare_path = found["val/mof/part_00000.aselmdb"]


In [ ]:
base_output = Path("/kaggle/working/phase3_material_base.json")
command = [sys.executable, str(root / "kaggle/phase3_material_pipeline.py"),
           "--train-parquet", train_path, "--val-target", str(target_path),
           "--val-bare", str(bare_path), "--output", str(base_output)]
frozen = root / "kaggle/frozen_targets_v1.json"
if frozen.is_file():
    command += ["--frozen-targets", str(frozen)]
subprocess.run(command, check=True, env=env)
base = json.loads(base_output.read_text())
assert len(base["validation_rows"]) == 138
assert len(base["validation_components"]) == 276
print("train pairs", len(base["train_rows"]), "validation pairs", len(base["validation_rows"]))
print(json.dumps({k: v["metrics"] for k, v in base["simple_baselines"].items()
                  if isinstance(v, dict) and "metrics" in v}, indent=2))


In [ ]:
from huggingface_hub import HfApi
ODAC_REVISION = HfApi(token=os.environ["HF_TOKEN"]).model_info("facebook/ODAC25").sha
checkpoint = hf_hub_download("facebook/ODAC25",
    "checkpoints/esen_sm_odac25_filtered.pt", revision=ODAC_REVISION, token=os.environ["HF_TOKEN"])
esen_output = Path("/kaggle/working/phase3_material_esen.json")
command = [sys.executable, str(root / "kaggle/phase3_material_pipeline.py"),
           "--train-parquet", train_path, "--val-target", str(target_path),
           "--val-bare", str(bare_path), "--base-input", str(base_output),
           "--esen-checkpoint", checkpoint, "--output", str(esen_output)]
subprocess.run(command, check=True, env=env)
esen = json.loads(esen_output.read_text())
print(json.dumps(esen["models"]["esen"].get("evaluation", {}), indent=2))


In [ ]:
uma_output = Path("/kaggle/working/phase3_material_uma.json")
command = [sys.executable, str(root / "kaggle/phase3_material_pipeline.py"),
           "--train-parquet", train_path, "--val-target", str(target_path),
           "--val-bare", str(bare_path), "--base-input", str(base_output),
           "--uma", "--output", str(uma_output)]
subprocess.run(command, check=True, env=env)
uma = json.loads(uma_output.read_text())
print(json.dumps(uma["models"]["uma"].get("evaluation", {}), indent=2))


In [ ]:
from importlib.metadata import version

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

result = base
result["models"] = {"esen": esen["models"]["esen"], "uma": uma["models"]["uma"]}
result["execution"] = {
    "python": sys.version, "platform": platform.platform(),
    "gpu": __import__("torch").cuda.get_device_name(0),
    "packages": {name: version(name) for name in
                 ("torch", "fairchem-core", "fairchem-data-odac", "ase", "numpy", "pyarrow")},
    "train": {"repository": TRAIN_REPO, "revision": TRAIN_REVISION,
              "path": TRAIN_FILE, "bytes": Path(train_path).stat().st_size,
              "sha256": digest(train_path)},
    "validation": {str(path.relative_to(data_root)): {"bytes": path.stat().st_size,
                    "sha256": digest(path)} for path in (target_path, bare_path)},
    "esen_checkpoint": {"revision": ODAC_REVISION, "filename": Path(checkpoint).name,
                        "bytes": Path(checkpoint).stat().st_size,
                        "sha256": digest(checkpoint)},
}
final_json = Path("/kaggle/working/phase3_material_results.json")
final_json.write_text(json.dumps(result, indent=2, allow_nan=False) + "\n")
archive = Path("/kaggle/working/ODAC25_Phase3_material_artifacts.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in (base_output, esen_output, uma_output, final_json):
        bundle.write(path, path.name)
print("artifact", archive, archive.stat().st_size)
print({name: {"failures": len(block["failures"]),
                    "seconds": block["elapsed_seconds"],
                    "metrics": block.get("evaluation")}
       for name, block in result["models"].items()})
